In [35]:
# import tools for selecting the correct python executable
import os
import sys

# use the same python executable for spark driver and workers
python_path = sys.executable

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

# import sparksession
from pyspark.sql import SparkSession

# create or reuse the spark session
spark = (
    SparkSession.builder
    .appName("dataset_joining")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "Europe/London")
    .config(
        "spark.pyspark.python",
        python_path
    )
    .config(
        "spark.pyspark.driver.python",
        python_path
        )
    .config(
        "spark.executorEnv.PYSPARK_PYTHON",
        python_path
    )
    .getOrCreate()
)

# make sure timetable and live times use uk local time
spark.conf.set(
    "spark.sql.session.timeZone",
    "Europe/London"
)

# reduce unnecessary spark messages
spark.sparkContext.setLogLevel("WARN")

print("python version:", sys.version.split()[0])
print("python path:", python_path)
print("pyspark version:", spark.version)
print(
    "shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)
print(
    "session time zone:",
    spark.conf.get("spark.sql.session.timeZone")
)
print("spark ui:", spark.sparkContext.uiWebUrl)

python version: 3.11.15
python path: /opt/anaconda3/envs/pyspark35/bin/python
pyspark version: 3.5.1
shuffle partitions: 4
session time zone: Europe/London
spark ui: http://192.168.1.8:4040


In [36]:
from pathlib import Path
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [37]:
#  folder where the notebook is running
current_folder = Path.cwd()
if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder
timetable_path = (
    project_root
    / "data"
    / "processed"
    / "timetable"
)
vehicle_path = (
    project_root
    / "data"
    / "processed"
    / "vehicle_locations"
)
joined_path = (
    project_root
    / "data"
    / "processed"
    / "joined_transport_data"
)
print("project root:", project_root)
print("timetable path:", timetable_path)
print("timetable path exists:", timetable_path.exists())
print("vehicle path:", vehicle_path)
print("vehicle path exists:", vehicle_path.exists())
print("joined output path:", joined_path)

project root: /Users/babitaadhikari/Desktop/bus-disruption-platform
timetable path: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/processed/timetable
timetable path exists: True
vehicle path: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/processed/vehicle_locations
vehicle path exists: True
joined output path: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/processed/joined_transport_data


## Load dataset

In [38]:
# load the processed timetable dataset
timetable_df = spark.read.parquet(
    str(timetable_path)
)
# load the processed vehicle location dataset
vehicle_df = spark.read.parquet(
    str(vehicle_path)
)
print(
    "timetable rows:",
    f"{timetable_df.count():,}"
)
print(
    "vehicle observations:",
    f"{vehicle_df.count():,}"
)
print(
    "timetable partitions:",
    timetable_df.rdd.getNumPartitions()
)
print(
    "vehicle partitions:",
    vehicle_df.rdd.getNumPartitions()
)

timetable rows: 5,370,199
vehicle observations: 5,824
timetable partitions: 8
vehicle partitions: 4


In [39]:
# display all timetable column names
print("timetable columns")
for column_name in timetable_df.columns:
    print(column_name)
# display all vehicle location column names
print("\nvehicle location columns")
for column_name in vehicle_df.columns:
    print(column_name)

timetable columns
agency_id
agency_name
agency_noc
route_id
route_short_name
route_long_name
service_id
trip_id
vehicle_journey_code
trip_headsign
direction_id
stop_id
stop_code
stop_name
stop_sequence
arrival_time
departure_time
arrival_seconds
departure_seconds
stop_lat
stop_lon
shape_id

vehicle location columns
source_file
snapshot_time
recorded_at_time
valid_until_time
operator_ref
line_ref
published_line_name
direction_ref
dated_vehicle_journey_ref
journey_pattern_ref
vehicle_ref
block_ref
origin_ref
destination_ref
destination_name
origin_aimed_departure_time
longitude
latitude
bearing
observation_date
observation_hour
observation_day_of_week
observation_age_seconds


In [40]:
# display the timetable schema
print("timetable schema")
timetable_df.printSchema()
# display the vehicle location schema
print("vehicle location schema")
vehicle_df.printSchema()

timetable schema
root
 |-- agency_id: string (nullable = true)
 |-- agency_name: string (nullable = true)
 |-- agency_noc: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- route_short_name: string (nullable = true)
 |-- route_long_name: string (nullable = true)
 |-- service_id: string (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- vehicle_journey_code: string (nullable = true)
 |-- trip_headsign: string (nullable = true)
 |-- direction_id: string (nullable = true)
 |-- stop_id: string (nullable = true)
 |-- stop_code: string (nullable = true)
 |-- stop_name: string (nullable = true)
 |-- stop_sequence: integer (nullable = true)
 |-- arrival_time: string (nullable = true)
 |-- departure_time: string (nullable = true)
 |-- arrival_seconds: integer (nullable = true)
 |-- departure_seconds: integer (nullable = true)
 |-- stop_lat: double (nullable = true)
 |-- stop_lon: double (nullable = true)
 |-- shape_id: string (nullable = true)

vehicle location sc

In [41]:
# define timetable columns required for joining
required_timetable_columns = [
    "agency_noc",
    "route_id",
    "route_short_name",
    "trip_id",
    "vehicle_journey_code",
    "direction_id",
    "stop_id",
    "stop_name",
    "stop_sequence",
    "arrival_seconds",
    "departure_seconds"
]
# define vehicle columns required for joining
required_vehicle_columns = [
    "source_file",
    "recorded_at_time",
    "operator_ref",
    "line_ref",
    "dated_vehicle_journey_ref",
    "direction_ref",
    "vehicle_ref",
    "origin_aimed_departure_time"
]
# find missing timetable columns
missing_timetable_columns = [
    column_name
    for column_name in required_timetable_columns
    if column_name not in timetable_df.columns
]
# find missing vehicle columns
missing_vehicle_columns = [
    column_name
    for column_name in required_vehicle_columns
    if column_name not in vehicle_df.columns
]
print(
    "missing timetable columns:",
    missing_timetable_columns
)
print(
    "missing vehicle columns:",
    missing_vehicle_columns
)
if missing_timetable_columns:
    raise ValueError(
        f"missing timetable columns {missing_timetable_columns}"
    )
if missing_vehicle_columns:
    raise ValueError(
        f"missing vehicle columns {missing_vehicle_columns}"
    )
print("all required joining columns are available")

missing timetable columns: []
missing vehicle columns: []
all required joining columns are available


In [42]:
# display sample timetable identifiers
print("timetable joining values")
timetable_df.select(
    "agency_noc",
    "route_short_name",
    "vehicle_journey_code",
    "direction_id"
).where(
    F.col("vehicle_journey_code").isNotNull()
).dropDuplicates().show(
    20,
    truncate=False
)
# display sample vehicle location identifiers
print("vehicle location joining values")
vehicle_df.select(
    "operator_ref",
    "line_ref",
    "dated_vehicle_journey_ref",
    "direction_ref"
).where(
    F.col("dated_vehicle_journey_ref").isNotNull()
).dropDuplicates().show(
    20,
    truncate=False
)

timetable joining values


+----------+----------------+--------------------+------------+
|agency_noc|route_short_name|vehicle_journey_code|direction_id|
+----------+----------------+--------------------+------------+
|TNXB      |35              |VJ225               |1           |
|TNXB      |997             |VJ22                |0           |
|DIAM      |21              |vj_11               |1           |
|TNXB      |4               |VJ297               |0           |
|TNXB      |94              |VJ176               |1           |
|NATX      |100             |VJ20                |1           |
|DIAM      |12A             |vj_7                |0           |
|TNXB      |X8              |VJ49                |0           |
|TNXB      |X4              |VJ377               |1           |
|SCGL      |33              |VJ23                |0           |
|TNXB      |79              |VJ316               |1           |
|TNXB      |X4              |VJ48                |1           |
|TNXB      |32              |VJ84       

In [43]:
# group stop level timetable records into one row for each trip
trip_level_df = (
    timetable_df
    .groupBy(
        "agency_noc",
        "route_id",
        "route_short_name",
        "service_id",
        "trip_id",
        "vehicle_journey_code",
        "trip_headsign",
        "direction_id"
    )
    .agg(
        F.min(
            "departure_seconds"
        ).alias(
            "scheduled_start_seconds"
        ),

        F.max(
            "arrival_seconds"
        ).alias(
            "scheduled_end_seconds"
        ),

        F.min(
            "stop_sequence"
        ).alias(
            "first_stop_sequence"
        ),

        F.max(
            "stop_sequence"
        ).alias(
            "last_stop_sequence"
        ),

        F.countDistinct(
            "stop_id"
        ).alias(
            "scheduled_stop_count"
        )
    )
)
print(
    "trip level timetable rows:",
    f"{trip_level_df.count():,}"
)
trip_level_df.show(
    10,
    truncate=False
)

trip level timetable rows: 135,003


26/07/14 10:24:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/14 10:24:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/14 10:24:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/14 10:24:36 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/14 10:24:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/14 10:24:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/14 10:24:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/14 10:24:38 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/07/14 10:24:39 WARN RowBasedKeyValueBatch: Calling spill() on

+----------+--------+----------------+----------+------------------------------------------+--------------------+------------------------------+------------+-----------------------+---------------------+-------------------+------------------+--------------------+
|agency_noc|route_id|route_short_name|service_id|trip_id                                   |vehicle_journey_code|trip_headsign                 |direction_id|scheduled_start_seconds|scheduled_end_seconds|first_stop_sequence|last_stop_sequence|scheduled_stop_count|
+----------+--------+----------------+----------+------------------------------------------+--------------------+------------------------------+------------+-----------------------+---------------------+-------------------+------------------+--------------------+
|TNXB      |1719805 |997             |620       |VJ006ad9cca356b4b9020bb8f1d151560e105c5585|VJ22                |Walsall Bus Station           |0           |38880                  |42780                |0    

In [44]:
# create a function that removes spaces and special characters
def compact_key(column_name):
    return F.upper(
        F.regexp_replace(
            F.trim(
                F.col(column_name)
            ),
            r"[^A-Za-z0-9]",
            ""
        )
    )
# create a function that extracts the final part of a journey reference
def journey_tail_key(column_name):
    raw_value = F.upper(
        F.trim(
            F.col(column_name)
        )
    )

    tail_value = F.regexp_extract(
        raw_value,
        r"([^:/|]+)$",
        1
    )
    return F.regexp_replace(
        tail_value,
        r"[^A-Z0-9]",
        ""
    )
print("join key functions created")

join key functions created


In [45]:
# create cleaned timetable identifiers
trip_prepared_df = (
    trip_level_df

    .withColumn(
        "timetable_operator_key",
        compact_key("agency_noc")
    )

    .withColumn(
        "timetable_line_key",
        compact_key("route_short_name")
    )

    .withColumn(
        "timetable_journey_full_key",
        compact_key("vehicle_journey_code")
    )

    .withColumn(
        "timetable_journey_tail_key",
        journey_tail_key(
            "vehicle_journey_code"
        )
    )

    .withColumn(
        "timetable_direction_key",
        compact_key("direction_id")
    )

    .withColumn(
        "scheduled_clock_seconds",
        F.pmod(
            F.col("scheduled_start_seconds"),
            F.lit(86400)
        )
    )
)
# cache because these keys will be checked several times
trip_prepared_df.cache()

print(
    "prepared timetable trips:",
    f"{trip_prepared_df.count():,}"
)

26/07/14 10:24:56 WARN CacheManager: Asked to cache already cached data.


prepared timetable trips: 135,003


In [46]:
# create cleaned vehicle location identifiers
vehicle_prepared_df = (
    vehicle_df
    .withColumn(
        "vehicle_operator_key",
        compact_key("operator_ref")
    )

    .withColumn(
        "vehicle_line_ref_key",
        compact_key("line_ref")
    )

    .withColumn(
        "vehicle_published_line_key",
        compact_key("published_line_name")
    )

    .withColumn(
        "vehicle_journey_full_key",
        compact_key(
            "dated_vehicle_journey_ref"
        )
    )

    .withColumn(
        "vehicle_journey_tail_key",
        journey_tail_key(
            "dated_vehicle_journey_ref"
        )
    )

    .withColumn(
        "vehicle_direction_key",
        compact_key("direction_ref")
    )

    .withColumn(
        "aimed_start_seconds",
        F.hour(
            F.col(
                "origin_aimed_departure_time"
            )
        ) * 3600
        + F.minute(
            F.col(
                "origin_aimed_departure_time"
            )
        ) * 60
        + F.second(
            F.col(
                "origin_aimed_departure_time"
            )
        )
    )
)
# create one unique identifier for every live observation
vehicle_prepared_df = (
    vehicle_prepared_df
    .withColumn(
        "observation_id",
        F.sha2(
            F.concat_ws(
                "||",

                F.coalesce(
                    F.col("source_file"),
                    F.lit("")
                ),

                F.coalesce(
                    F.col("vehicle_ref"),
                    F.lit("")
                ),

                F.coalesce(
                    F.col(
                        "recorded_at_time"
                    ).cast("string"),
                    F.lit("")
                ),

                F.coalesce(
                    F.col("line_ref"),
                    F.lit("")
                )
            ),
            256
        )
    )
)
# cache because these keys will be reused
vehicle_prepared_df.cache()
print(
    "prepared vehicle observations:",
    f"{vehicle_prepared_df.count():,}"
)

prepared vehicle observations: 5,824


26/07/14 10:24:57 WARN CacheManager: Asked to cache already cached data.


In [ ]:
# display sample timetable join keys
print("prepared timetable keys")
trip_prepared_df.select(
    "agency_noc",
    "timetable_operator_key",
    "route_short_name",
    "timetable_line_key",
    "vehicle_journey_code",
    "timetable_journey_full_key",
    "timetable_journey_tail_key"
).where(
    F.col(
        "timetable_journey_full_key"
    ).isNotNull()
).dropDuplicates().show(
    20,
    truncate=False
)

# display sample vehicle join keys
print("prepared vehicle keys")

vehicle_prepared_df.select(
    "operator_ref",
    "vehicle_operator_key",
    "line_ref",
    "vehicle_line_ref_key",
    "published_line_name",
    "vehicle_published_line_key",
    "dated_vehicle_journey_ref",
    "vehicle_journey_full_key",
    "vehicle_journey_tail_key"
).where(
    F.col(
        "vehicle_journey_full_key"
    ).isNotNull()
).dropDuplicates().show(
    20,
    truncate=False
)

prepared timetable keys


In [ ]:
# create a function that counts matching distinct values
def count_key_overlap(
    left_df,
    left_column,
    right_df,
    right_column
):
    left_keys = (
        left_df
        .select(
            F.col(left_column).alias(
                "join_key"
            )
        )
        .filter(
            F.col("join_key").isNotNull()
            & (F.col("join_key") != "")
        )
        .distinct()
    )

    right_keys = (
        right_df
        .select(
            F.col(right_column).alias(
                "join_key"
            )
        )
        .filter(
            F.col("join_key").isNotNull()
            & (F.col("join_key") != "")
        )
        .distinct()
    )

    overlap_count = (
        left_keys
        .join(
            right_keys,
            on="join_key",
            how="inner"
        )
        .count()
    )

    return overlap_count
print("key overlap function created")

In [ ]:
# compare timetable routes with vehicle line references
line_ref_overlap = count_key_overlap(
    trip_prepared_df,
    "timetable_line_key",
    vehicle_prepared_df,
    "vehicle_line_ref_key"
)
# compare timetable routes with published line names
published_line_overlap = count_key_overlap(
    trip_prepared_df,
    "timetable_line_key",
    vehicle_prepared_df,
    "vehicle_published_line_key"
)
# compare complete journey references
full_journey_overlap = count_key_overlap(
    trip_prepared_df,
    "timetable_journey_full_key",
    vehicle_prepared_df,
    "vehicle_journey_full_key"
)
# compare final parts of journey references
tail_journey_overlap = count_key_overlap(
    trip_prepared_df,
    "timetable_journey_tail_key",
    vehicle_prepared_df,
    "vehicle_journey_tail_key"
)
# compare operator references
operator_overlap = count_key_overlap(
    trip_prepared_df,
    "timetable_operator_key",
    vehicle_prepared_df,
    "vehicle_operator_key"
)
print(
    "line ref overlap:",
    line_ref_overlap
)
print(
    "published line overlap:",
    published_line_overlap
)
print(
    "full journey overlap:",
    full_journey_overlap
)
print(
    "tail journey overlap:",
    tail_journey_overlap
)

print(
    "operator overlap:",
    operator_overlap
)

In [ ]:
# choose the vehicle route column with the most overlap
if line_ref_overlap >= published_line_overlap:
    selected_vehicle_line_column = (
        "vehicle_line_ref_key"
    )

    selected_line_method = "line ref"

else:
    selected_vehicle_line_column = (
        "vehicle_published_line_key"
    )

    selected_line_method = (
        "published line name"
    )
# choose the journey column with the most overlap
if full_journey_overlap >= tail_journey_overlap:
    selected_timetable_journey_column = (
        "timetable_journey_full_key"
    )

    selected_vehicle_journey_column = (
        "vehicle_journey_full_key"
    )

    selected_journey_method = (
        "full journey reference"
    )

else:
    selected_timetable_journey_column = (
        "timetable_journey_tail_key"
    )

    selected_vehicle_journey_column = (
        "vehicle_journey_tail_key"
    )

    selected_journey_method = (
        "journey reference tail"
    )


print(
    "selected line method:",
    selected_line_method
)

print(
    "selected vehicle line column:",
    selected_vehicle_line_column
)

print(
    "selected journey method:",
    selected_journey_method
)

print(
    "selected timetable journey column:",
    selected_timetable_journey_column
)

print(
    "selected vehicle journey column:",
    selected_vehicle_journey_column
)

In [ ]:
# create aliases for clear join references
vehicle_alias = vehicle_prepared_df.alias("v")

timetable_alias = trip_prepared_df.alias("t")


# define the vehicle journey key
vehicle_journey_value = F.col(
    f"v.{selected_vehicle_journey_column}"
)

# define the timetable journey key
timetable_journey_value = F.col(
    f"t.{selected_timetable_journey_column}"
)

# define the vehicle line key
vehicle_line_value = F.col(
    f"v.{selected_vehicle_line_column}"
)

# define the timetable line key
timetable_line_value = F.col(
    "t.timetable_line_key"
)


# match records using route and journey references
join_condition = (
    vehicle_journey_value.isNotNull()
    & timetable_journey_value.isNotNull()
    & (vehicle_journey_value != "")
    & (timetable_journey_value != "")
    & (
        vehicle_journey_value
        == timetable_journey_value
    )
    & (
        vehicle_line_value
        == timetable_line_value
    )
)


# keep every vehicle observation using a left join
candidate_matches_df = (
    vehicle_alias
    .join(
        timetable_alias,
        on=join_condition,
        how="left"
    )
    .select(
        "v.*",

        F.col("t.trip_id").alias(
            "matched_trip_id"
        ),

        F.col("t.service_id").alias(
            "matched_service_id"
        ),

        F.col("t.route_id").alias(
            "matched_route_id"
        ),

        F.col("t.route_short_name").alias(
            "matched_route_short_name"
        ),

        F.col("t.vehicle_journey_code").alias(
            "matched_vehicle_journey_code"
        ),

        F.col("t.direction_id").alias(
            "matched_direction_id"
        ),

        F.col("t.trip_headsign").alias(
            "matched_trip_headsign"
        ),

        F.col("t.scheduled_start_seconds").alias(
            "scheduled_start_seconds"
        ),

        F.col("t.scheduled_clock_seconds").alias(
            "scheduled_clock_seconds"
        ),

        F.col("t.scheduled_end_seconds").alias(
            "scheduled_end_seconds"
        ),

        F.col("t.scheduled_stop_count").alias(
            "scheduled_stop_count"
        ),

        F.col("t.timetable_operator_key").alias(
            "matched_operator_key"
        ),

        F.col("t.timetable_direction_key").alias(
            "matched_direction_key"
        )
    )
)

print(
    "candidate match rows:",
    f"{candidate_matches_df.count():,}"
)

In [ ]:
# calculate whether operator values match
candidate_matches_df = (
    candidate_matches_df

    .withColumn(
        "operator_key_match",
        F.when(
            F.col("vehicle_operator_key")
            == F.col("matched_operator_key"),
            1
        ).otherwise(0)
    )

    .withColumn(
        "direction_key_match",
        F.when(
            F.col("vehicle_direction_key")
            == F.col("matched_direction_key"),
            1
        ).otherwise(0)
    )
)


# calculate the direct clock time difference
candidate_matches_df = (
    candidate_matches_df

    .withColumn(
        "raw_time_difference_seconds",
        F.abs(
            F.col("scheduled_clock_seconds")
            - F.col("aimed_start_seconds")
        )
    )

    .withColumn(
        "schedule_time_difference_seconds",
        F.when(
            F.col(
                "raw_time_difference_seconds"
            ).isNotNull(),

            F.least(
                F.col(
                    "raw_time_difference_seconds"
                ),

                F.lit(86400)
                - F.col(
                    "raw_time_difference_seconds"
                )
            )
        )
    )
)

print("candidate match quality values created")

In [ ]:
# define how timetable candidates should be ranked
best_match_window = (
    Window
    .partitionBy("observation_id")
    .orderBy(
        F.col(
            "operator_key_match"
        ).desc(),

        F.col(
            "direction_key_match"
        ).desc(),

        F.col(
            "schedule_time_difference_seconds"
        ).asc_nulls_last(),

        F.col(
            "matched_trip_id"
        ).asc_nulls_last()
    )
)


# rank the candidates and keep one match per observation
joined_df = (
    candidate_matches_df

    .withColumn(
        "match_rank",
        F.row_number().over(
            best_match_window
        )
    )

    .filter(
        F.col("match_rank") == 1
    )

    .drop(
        "match_rank",
        "raw_time_difference_seconds"
    )
)

print(
    "joined observation rows:",
    f"{joined_df.count():,}"
)

In [ ]:
# create match status and confidence columns
joined_df = (
    joined_df

    .withColumn(
        "is_timetable_matched",
        F.when(
            F.col(
                "matched_trip_id"
            ).isNotNull(),
            1
        ).otherwise(0)
    )

    .withColumn(
        "match_quality",
        F.when(
            F.col(
                "matched_trip_id"
            ).isNull(),
            "unmatched"
        )

        .when(
            (
                F.col(
                    "schedule_time_difference_seconds"
                ) <= 900
            )
            & (
                F.col(
                    "operator_key_match"
                ) == 1
            ),
            "high"
        )

        .when(
            F.col(
                "schedule_time_difference_seconds"
            ) <= 1800,
            "medium"
        )

        .when(
            F.col(
                "schedule_time_difference_seconds"
            ).isNull(),
            "journey and line only"
        )

        .otherwise(
            "low"
        )
    )

    .withColumn(
        "selected_line_method",
        F.lit(
            selected_line_method
        )
    )

    .withColumn(
        "selected_journey_method",
        F.lit(
            selected_journey_method
        )
    )
)

print("match status columns created")

In [ ]:
# count original vehicle observations
original_vehicle_count = (
    vehicle_prepared_df.count()
)

# count final joined observations
final_joined_count = (
    joined_df.count()
)

print(
    "original vehicle observations:",
    f"{original_vehicle_count:,}"
)

print(
    "final joined observations:",
    f"{final_joined_count:,}"
)

# stop when joining creates or removes observation rows
if final_joined_count != original_vehicle_count:
    raise ValueError(
        "final joined rows do not match vehicle observations"
    )

print("one final row exists for every vehicle observation")

In [ ]:
# count successfully matched observations
matched_vehicle_observations = (
    joined_df
    .filter(
        F.col(
            "is_timetable_matched"
        ) == 1
    )
    .count()
)

# calculate unmatched observations
unmatched_vehicle_observations = (
    final_joined_count
    - matched_vehicle_observations
)

# calculate the match percentage
match_rate = (
    matched_vehicle_observations
    / final_joined_count
    * 100
    if final_joined_count > 0
    else 0
)

print(
    "total vehicle observations:",
    f"{final_joined_count:,}"
)

print(
    "matched observations:",
    f"{matched_vehicle_observations:,}"
)

print(
    "unmatched observations:",
    f"{unmatched_vehicle_observations:,}"
)

print(
    "join match rate:",
    f"{match_rate:.2f}%"
)

In [ ]:
# count observations in every match quality category
match_quality_summary = (
    joined_df
    .groupBy(
        "match_quality"
    )
    .agg(
        F.count("*").alias(
            "observation_count"
        )
    )
    .orderBy(
        F.desc(
            "observation_count"
        )
    )
)

match_quality_summary.show(
    truncate=False
)

In [ ]:
# display sample successfully matched observations
joined_df.filter(
    F.col(
        "is_timetable_matched"
    ) == 1
).select(
    "vehicle_ref",
    "operator_ref",
    "line_ref",
    "dated_vehicle_journey_ref",
    "recorded_at_time",
    "origin_aimed_departure_time",
    "matched_trip_id",
    "matched_route_short_name",
    "matched_vehicle_journey_code",
    "scheduled_start_seconds",
    "schedule_time_difference_seconds",
    "operator_key_match",
    "direction_key_match",
    "match_quality"
).show(
    20,
    truncate=False
)

In [ ]:
# display sample unmatched observations
joined_df.filter(
    F.col(
        "is_timetable_matched"
    ) == 0
).select(
    "source_file",
    "vehicle_ref",
    "operator_ref",
    "line_ref",
    "published_line_name",
    "dated_vehicle_journey_ref",
    "direction_ref",
    "recorded_at_time"
).show(
    20,
    truncate=False
)

In [ ]:
# distribute the joined dataset across four route based partitions
joined_df = joined_df.repartition(
    4,
    "line_ref"
)

# cache the dataset for later analysis
joined_df.cache()

# trigger spark processing
joined_count = joined_df.count()

print(
    "final joined rows:",
    f"{joined_count:,}"
)

print(
    "joined partitions:",
    joined_df.rdd.getNumPartitions()
)

In [ ]:
# register the joined dataframe as a temporary sql table
joined_df.createOrReplaceTempView(
    "joined_transport_data"
)

print("temporary sql view created")
print("view name: joined_transport_data")

In [ ]:
# calculate match rates for each bus line
join_line_summary = spark.sql("""
    SELECT
        line_ref,
        published_line_name,
        COUNT(*) AS total_observations,
        SUM(is_timetable_matched) AS matched_observations,
        ROUND(
            SUM(is_timetable_matched) * 100.0
            / COUNT(*),
            2
        ) AS match_rate_percent
    FROM joined_transport_data
    GROUP BY
        line_ref,
        published_line_name
    ORDER BY
        total_observations DESC
    LIMIT 20
""")

join_line_summary.show(
    truncate=False
)

In [ ]:
# display how spark plans the joining operations
joined_df.explain(
    mode="formatted"
)

In [ ]:
# create the joined output folder when it does not exist
joined_path.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "joined output folder:",
    joined_path
)

In [ ]:
# save the joined transport dataset as parquet
(
    joined_df.write
    .mode("overwrite")
    .parquet(
        str(joined_path)
    )
)

print("joined transport data saved")
print("parquet location:", joined_path)

In [ ]:
# read the saved parquet dataset
saved_joined_df = spark.read.parquet(
    str(joined_path)
)

# count the saved rows
saved_joined_count = (
    saved_joined_df.count()
)

print(
    "original joined rows:",
    f"{joined_count:,}"
)

print(
    "saved joined rows:",
    f"{saved_joined_count:,}"
)

print(
    "saved partitions:",
    saved_joined_df.rdd.getNumPartitions()
)

# stop when the saved row count does not match
if saved_joined_count != joined_count:
    raise ValueError(
        "saved joined count does not match original count"
    )

print("joined parquet verification successful")

In [ ]:
# define the csv output folder
joined_csv_path = (
    project_root
    / "data"
    / "processed"
    / "joined_transport_data_csv"
)

# export the joined dataset as one csv part file
(
    saved_joined_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option(
        "header",
        True
    )
    .csv(
        str(joined_csv_path)
    )
)

print("joined csv dataset saved")
print("csv location:", joined_csv_path)

In [ ]:
# display the final dataset joining results
print("dataset joining summary")
print("-" * 50)

print(
    "trip level timetable rows:",
    f"{trip_level_df.count():,}"
)

print(
    "vehicle observations:",
    f"{original_vehicle_count:,}"
)

print(
    "matched observations:",
    f"{matched_vehicle_observations:,}"
)

print(
    "unmatched observations:",
    f"{unmatched_vehicle_observations:,}"
)

print(
    "join match rate:",
    f"{match_rate:.2f}%"
)

print(
    "selected line method:",
    selected_line_method
)

print(
    "selected journey method:",
    selected_journey_method
)

print(
    "final partitions:",
    joined_df.rdd.getNumPartitions()
)

print("main saved format: parquet")
print("parquet location:", joined_path)
print("csv location:", joined_csv_path)